# The Gemma 4 Developer Agent Competition - Starter Notebook

This notebook demonstrates how to load, test, and package an autonomous SWE agent for **The Gemma 4 Developer Agent Competition** on Kaggle.

### Overview
- **Model**: `gemma-4-31b-it-qat-w4a16-ct`
- **Format**: Declarative configuration (`agent.yaml`) with prompts, tools, and optional LoRA adapters.
- **Harness**: `swegemma` evaluation toolkit.

## 1. Environment Setup

In [ ]:
!pip install --upgrade pip
!pip install swegemma pyyaml

## 2. Inspect Agent Configuration (`agent.yaml`)

The competition requires an `agent.yaml` at the root of `submission.zip`.

In [ ]:
%%writefile agent.yaml
name: gemma_developer_agent
model: gemma-4-31b-it-qat-w4a16-ct
description: Autonomous software engineering agent that diagnoses, edits, and verifies fixes for repository issues.
instruction: !include prompts/system.md
generate_content_config: !include configs/sampling.yaml
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph


## 3. Define System Prompt & Sampling Config

In [ ]:
import os
os.makedirs('prompts', exist_ok=True)
os.makedirs('configs', exist_ok=True)

with open('prompts/system.md', 'w') as f:
    f.write('''You are an expert autonomous software engineer operating inside a repository sandbox.
Your mission is to resolve the given issue report by locating the root cause, applying the minimal necessary fix, verifying the fix with tests, and submitting a clean git patch.

Workflow:
1. Exploration & Localization: Search code, locate culprits with read_file.
2. Reproduction: Reproduce issue with test command via run_command.
3. Surgical Edit: Apply minimal fix using edit_file.
4. Verification: Run tests via run_command to verify pass.
5. Submission: Call submit_patch.
''')

with open('configs/sampling.yaml', 'w') as f:
    f.write('''temperature: 0.2
top_p: 0.95
top_k: 40
max_output_tokens: 4096
stop_sequences:
  - "<end_of_turn>"
''')
print('Configuration files created.')

## 4. Packaging `submission.zip`

In [ ]:
import zipfile

with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('agent.yaml', arcname='agent.yaml')
    z.write('prompts/system.md', arcname='prompts/system.md')
    z.write('configs/sampling.yaml', arcname='configs/sampling.yaml')

print('submission.zip generated successfully!')